# 02 — Training Loop

**Prerequisites**:
- `00_csi_pipeline.ipynb` — UNIFIED.jsonl + model definition
- `01_tokenization.ipynb` — token cache in `datasets/token_cache/`

### What this notebook does
1. **Load** token cache + config.yaml
2. **Init** GraphCodeBERTLoRACWEModel (LoRA, 0.24% trainable)
3. **Train** — AdamW + linear warmup, cross-entropy on 8-class CWE head
4. **Validate** — F1 / precision / recall per CWE class + macro-avg after each epoch
5. **Checkpoint** — save best model by macro F1
6. **Log** — metrics to JSON file (+ Google Drive if on Colab)

### Platform support
| Platform | Backend | Notes |
|---|---|---|
| Google Colab T4/A100 | **CUDA** | Recommended — full speed |
| Apple Silicon (M1/M2/M3) | **MPS** | ~3–5× slower than CUDA |
| CPU | **CPU** | Very slow — dev/debug only |

## 0 — Install Dependencies

In [4]:
import subprocess, sys, platform

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in __import__("os").environ

print(f"Platform      : {platform.system()} {platform.machine()}")
print(f"Apple Silicon : {IS_APPLE_SILICON}")
print(f"Colab         : {IS_COLAB}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=4.40",
        "peft>=0.10",
        "torch",
        "pyyaml",
        "scikit-learn",
        "tqdm",
    ],
    check=False,
)
print("Done.")

Platform      : Linux x86_64
Apple Silicon : False
Colab         : True
Done.


## 1 — Paths, Config & Device

In [5]:
import os, json, platform, sys
from pathlib import Path
import yaml
import torch

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"

try:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    IS_COLAB = True
    print("Colab — Drive mounted")
except ImportError:
    BASE_DIR = Path(os.path.dirname(os.path.abspath("__file__")))
    if not (BASE_DIR / "datasets").exists():
        BASE_DIR = Path.cwd()
    IS_COLAB = False
    print(f"Local — BASE_DIR: {BASE_DIR}")

# Load config
cfg_path = BASE_DIR / "config.yaml"
assert cfg_path.exists(), f"Missing config.yaml at {cfg_path}"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Paths
TOKEN_CACHE_DIR = BASE_DIR / cfg["token_cache_dir"]
CHECKPOINT_DIR = BASE_DIR / cfg["checkpoint_dir"]
LOG_DIR = BASE_DIR / cfg["log_dir"]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = cfg["max_length"]
CACHE_FILE = TOKEN_CACHE_DIR / f"tokens_maxlen{MAX_LENGTH}.pt"
assert (
    CACHE_FILE.exists()
), f"Missing cache: {CACHE_FILE} — run 01_tokenization.ipynb first"

# Device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif IS_APPLE_SILICON and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device          : {DEVICE}")
print(f"CACHE_FILE      : {CACHE_FILE}")
print(f"CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
print(f"LOG_DIR         : {LOG_DIR}")

# Reproducibility
import random, numpy as np

SEED = cfg["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab — Drive mounted
Device          : cuda
CACHE_FILE      : /content/drive/MyDrive/CSI_Project/datasets/token_cache/tokens_maxlen512.pt
CHECKPOINT_DIR  : /content/drive/MyDrive/CSI_Project/checkpoints
LOG_DIR         : /content/drive/MyDrive/CSI_Project/logs


## 2 — Load Token Cache + Build DataLoaders

In [6]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np

# ── Copy dataset + dataloader code from 01_tokenization.ipynb ────────────────
# (self-contained so this notebook runs independently)


class VulnerabilityDataset(Dataset):
    def __init__(self, cache: dict, split: str):
        if split == "all":
            indices = list(range(len(cache["split_origins"])))
        else:
            indices = [i for i, s in enumerate(cache["split_origins"]) if s == split]
        self.input_ids = cache["input_ids"][indices]
        self.attention_mask = cache["attention_mask"][indices]
        self.cwe_labels = cache["cwe_labels"][indices]
        self.binary_labels = cache["binary_labels"][indices]
        self.global_ids = cache["global_ids"][indices]
        self.split = split

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "cwe_label": self.cwe_labels[idx],
            "binary_label": self.binary_labels[idx],
            "global_id": self.global_ids[idx],
        }


def make_balanced_sampler(dataset):
    labels = dataset.cwe_labels.numpy()
    class_counts = np.bincount(labels, minlength=8)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float),
        num_samples=len(labels),
        replacement=True,
    )


# Load cache
cache = torch.load(CACHE_FILE, weights_only=True)
print(f'Cache loaded: {cache["num_records"]:,} records')

BATCH_SIZE = cfg["batch_size"]
train_ds = VulnerabilityDataset(cache, split="train")
val_ds = VulnerabilityDataset(cache, split="val")

sampler = make_balanced_sampler(train_ds)
train_loader = DataLoader(
    train_ds,
    batch_size=cfg["batch_size"],
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg["batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

print(f"Train batches : {len(train_loader):,}  ({len(train_ds):,} samples)")
print(f"Val batches   : {len(val_loader):,}  ({len(val_ds):,} samples)")

Cache loaded: 14,522 records
Train batches : 813  (12,994 samples)
Val batches   : 96  (1,528 samples)


## 3 — Model, Optimizer & Scheduler

In [8]:
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model
from typing import Dict

CWE_8_CLASSES = [
    "CWE-077",
    "CWE-601",
    "CWE-022",
    "CWE-094",
    "CWE-089",
    "CWE-352",
    "CWE-079",
    "unknown",
]
CWE_TO_INDEX = {cwe: i for i, cwe in enumerate(CWE_8_CLASSES)}
INDEX_TO_CWE = {i: cwe for cwe, i in CWE_TO_INDEX.items()}


class CWEClassificationHead(nn.Module):
    def __init__(self, hidden_size: int, num_classes: int = 8, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        return self.classifier(self.dropout(x))


class GraphCodeBERTLoRACWEModel(nn.Module):
    def __init__(
        self,
        model_name: str = "microsoft/graphcodebert-base",
        num_cwe_classes: int = 8,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.1,
    ):
        super().__init__()
        encoder = AutoModel.from_pretrained(model_name)
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["query", "value"],
            bias="none",
        )
        self.encoder = get_peft_model(encoder, lora_cfg)
        self.cwe_head = CWEClassificationHead(
            self.encoder.config.hidden_size, num_cwe_classes
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, cwe_labels=None):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = enc.last_hidden_state[:, 0, :]  # (B, H)
        logits = self.cwe_head(cls_repr)  # (B, 8)
        result = {"logits": logits}
        if cwe_labels is not None:
            result["loss"] = self.loss_fn(logits, cwe_labels)
        return result


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total, 100 * trainable / total


# Init model
model = GraphCodeBERTLoRACWEModel(
    model_name=cfg["model_name"],
    num_cwe_classes=cfg["num_cwe_classes"],
    lora_r=cfg["lora_r"],
    lora_alpha=cfg["lora_alpha"],
    lora_dropout=cfg["lora_dropout"],
).to(DEVICE)

trainable, total, pct = count_params(model)
print(f"Model loaded to {DEVICE}")
print(f"Trainable: {trainable:,} / {total:,}  ({pct:.2f}%)")

# Optimizer + scheduler
EPOCHS = cfg["epochs"]
LR = cfg["learning_rate"]
WARMUP_RATIO = cfg["warmup_ratio"]

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=cfg["weight_decay"],
)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Epochs       : {EPOCHS}")
print(f"Total steps  : {total_steps:,}")
print(f"Warmup steps : {warmup_steps:,}")
print(f"LR           : {LR}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded to cuda
Trainable: 301,064 / 124,946,696  (0.24%)
Epochs       : 5
Total steps  : 4,065
Warmup steps : 406
LR           : 2e-05


## 4 — Train & Validate

In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)
from tqdm.auto import tqdm
import time

MAX_GRAD_NORM = cfg["max_grad_norm"]
LOG_EVERY = cfg["log_every_n_steps"]
SAVE_EVERY = cfg["save_every_n_steps"]
PATIENCE = cfg["early_stopping_patience"]

CHECKPOINT_DIR = Path(cfg["checkpoint_dir"])
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT = CHECKPOINT_DIR / "best_model.pt"

# ── Auto-resume from checkpoint ──────────────────────────────────────────────
BEST_F1 = 0.0
start_epoch = 1
no_improve = 0
history = []

if BEST_CKPT.exists():
    print(f"Found checkpoint: {BEST_CKPT}")
    checkpoint = torch.load(BEST_CKPT, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    BEST_F1 = checkpoint["val_f1"]
    start_epoch = checkpoint["epoch"] + 1
    print(f"✓ Resumed from epoch {start_epoch}, best F1={BEST_F1:.4f}")
else:
    print("No checkpoint found. Starting fresh.")


def validate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            cwe_labels = batch["cwe_label"].to(device)
            out = model(input_ids, attn_mask, cwe_labels=cwe_labels)
            total_loss += out["loss"].item()
            preds = out["logits"].argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(cwe_labels.cpu().tolist())
            n_batches += 1
    avg_loss = total_loss / n_batches
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_prec = precision_score(
        all_labels, all_preds, average="macro", zero_division=0
    )
    macro_rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, macro_f1, macro_prec, macro_rec, all_preds, all_labels


global_step = 0

for epoch in range(start_epoch, cfg["epochs"] + 1):
    model.train()
    epoch_loss = 0.0
    step_loss = 0.0
    t0 = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg['epochs']}", leave=True)
    for step, batch in enumerate(pbar, 1):
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        cwe_labels = batch["cwe_label"].to(DEVICE)

        optimizer.zero_grad()
        out = model(input_ids, attn_mask, cwe_labels=cwe_labels)
        loss = out["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()), MAX_GRAD_NORM
        )
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        step_loss += loss.item()
        global_step += 1

        if step % LOG_EVERY == 0:
            avg = step_loss / LOG_EVERY
            lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({"loss": f"{avg:.4f}", "lr": f"{lr:.2e}"})
            step_loss = 0.0

        if global_step % SAVE_EVERY == 0:
            ckpt_path = CHECKPOINT_DIR / f"step_{global_step}.pt"
            torch.save(
                {
                    "step": global_step,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                },
                ckpt_path,
            )
            print(f"  Saved step checkpoint: {ckpt_path}")

    train_loss = epoch_loss / len(train_loader)

    val_loss, val_f1, val_prec, val_rec, preds, labels = validate(
        model, val_loader, DEVICE
    )
    elapsed = time.time() - t0

    print(
        f"\nEpoch {epoch:02d}  "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}  "
        f"val_F1={val_f1:.4f}  "
        f"val_P={val_prec:.4f}  "
        f"val_R={val_rec:.4f}  "
        f"({elapsed:.0f}s)"
    )

    print(
        classification_report(
            labels, preds, target_names=CWE_8_CLASSES, zero_division=0
        )
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "val_f1": round(val_f1, 4),
        "val_prec": round(val_prec, 4),
        "val_rec": round(val_rec, 4),
        "elapsed_s": round(elapsed, 1),
    }
    history.append(epoch_record)

    if val_f1 > BEST_F1:
        BEST_F1 = val_f1
        no_improve = 0
        torch.save(
            {
                "epoch": epoch,
                "val_f1": val_f1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": cfg,
            },
            BEST_CKPT,
        )
        print(f"  New best F1={val_f1:.4f} — saved to {BEST_CKPT}")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nTraining complete. Best val F1 = {BEST_F1:.4f}")
print(f"History: {history}")

No checkpoint found. Starting fresh.


Epoch 1/5:   0%|          | 0/813 [00:00<?, ?it/s]

  Saved step checkpoint: checkpoints/step_200.pt
  Saved step checkpoint: checkpoints/step_400.pt
  Saved step checkpoint: checkpoints/step_600.pt


## 5 — Save Metrics Log

In [ ]:
import datetime

log = {
    "run_date": datetime.datetime.now().isoformat(),
    "device": DEVICE,
    "model_name": cfg["model_name"],
    "best_val_f1": BEST_F1,
    "config": cfg,
    "history": history,
}

log_path = LOG_DIR / f'run_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

print(f"Metrics saved: {log_path}")
print(f"Best val F1  : {BEST_F1:.4f}")
print(f"Best ckpt    : {BEST_CKPT}")

# Print history table
print(
    f'\n{"Epoch":>5} {"TrainLoss":>10} {"ValLoss":>10} {"ValF1":>8} {"ValP":>8} {"ValR":>8}'
)
for r in history:
    print(
        f'{r["epoch"]:>5} {r["train_loss"]:>10.4f} {r["val_loss"]:>10.4f} '
        f'{r["val_f1"]:>8.4f} {r["val_prec"]:>8.4f} {r["val_rec"]:>8.4f}'
    )

## 6 — Load Best Checkpoint (inference ready)

In [ ]:
# Load best checkpoint for inference / further evaluation
ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

print(f"Loaded best checkpoint")
print(f'  epoch   : {ckpt["epoch"]}')
print(f'  val_f1  : {ckpt["val_f1"]:.4f}')

# Quick inference example
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])

sample_code = (
    "def get_user(uid): return db.execute('SELECT * FROM users WHERE id=' + uid)"
)
enc = tokenizer(
    sample_code,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt",
)
enc = {k: v.to(DEVICE) for k, v in enc.items()}

with torch.no_grad():
    out = model(enc["input_ids"], enc["attention_mask"])
    pred = out["logits"].argmax(dim=-1).item()
    probs = torch.softmax(out["logits"], dim=-1)[0].tolist()

print(f"\nSample prediction:")
print(f"  code : {sample_code[:60]}...")
print(f"  pred : {INDEX_TO_CWE[pred]}  (index {pred})")
print(f"  top3 :")
top3 = sorted(enumerate(probs), key=lambda x: -x[1])[:3]
for idx, prob in top3:
    print(f"    {INDEX_TO_CWE[idx]:<12} {prob:.4f}")